# Entrenamiento en la nube (Kaggle / Colab)

Ejecuta los experimentos de la lista de punta a punta: clona el repositorio, descarga y prepara los datos, entrena, evalúa y empaqueta los resultados (`models/` y `experiments/`) en un zip por experimento.

**Kaggle:** si algún experimento usa Toloka (`toloka`, `combined` o `combined_energia`), adjuntar ese dataset con *Add Input* (buscar "yandextoloka water meters"; una ejecución en segundo plano no puede adjuntarlo sola). Activar *GPU* e *Internet* en *Settings*, cargar el Secret `ROBOFLOW_API_KEY` (*Add-ons → Secrets*) y ejecutar con *Save Version → Save & Run All* para que corra en segundo plano.

**Colab:** *Entorno de ejecución → GPU* y cargar el Secret `ROBOFLOW_API_KEY`.

In [1]:
# Experimentos a ejecutar, en orden (ver experiments/README.md).
# trainer: "yolo" (Ultralytics, por defecto), "maskrcnn" (torchvision) o "eval_coco"
#          (solo evalúa con métricas COCO un modelo ya versionado en models/).
# task: "segment" (etapa 1, visor) o "detect" (etapa 2, dígitos).
# dataset: "toloka" (solo Toloka), "combined" (Toloka + yfzw4 con visor derivado)
#          "digits" (dígitos de yfzw4 sobre el visor rectificado)
#          "digits_mix" (yfzw4 + dígitos LCD de energía: meter-digits-5srka y meter-digits-frvqn)
#          o "combined_energia" (combined + visores de energía CEML revisados, sin la muestra de test).
# fliplr: probabilidad de espejado horizontal; 0 para dígitos (un 2 espejado parece un 5).
# Cada experimento se empaqueta al terminar, así un corte por tiempo no pierde los anteriores:
# por eso van primero los rápidos y al final Mask R-CNN, que es el más largo.
EXPERIMENTS = [
    # E2: segmentador del visor con agua y energía, mismos hiperparámetros que B
    {"name": "toloka_yfzw4_energia_yolov8n_seg", "task": "segment", "dataset": "combined_energia",
     "model": "yolov8n-seg.pt", "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},
]

# Ya ejecutados (resultados en experiments/README.md):
# {"name": "digits_mix_yolov8n", "task": "detect", "dataset": "digits_mix", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # E1
# {"name": "toloka_yfzw4_yolov8n_seg", "trainer": "eval_coco", "task": "segment", "dataset": "combined",
#  "imgsz": 640},                                                                  # B con métrica COCO
# {"name": "digits_yolo11n", "task": "detect", "dataset": "digits", "model": "yolo11n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D2
# {"name": "toloka_yfzw4_maskrcnn", "trainer": "maskrcnn", "task": "segment", "dataset": "combined",
#  "epochs": 10, "batch": 4, "lr": 0.005},                                         # S4
# {"name": "toloka_yfzw4_yolov8n_seg", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},       # B
# {"name": "toloka_yfzw4_yolov8n_seg_1024", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 1024, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},      # C
# {"name": "digits_yolov8n", "task": "detect", "dataset": "digits", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D1

In [2]:
# Configuración del entorno: Kaggle, Colab o local.
import os
import subprocess
import sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/martinlacheski/ceia-vpc-2.git"
REPO_BRANCH = "medidores-energia"  # rama con los scripts de energía


def run(cmd: list[str]) -> None:
    """Ejecuta un comando mostrando su salida y falla si termina con error."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if IN_KAGGLE or IN_COLAB:
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content")
    if not Path("ceia-vpc-2").exists():
        run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL])
    os.chdir("ceia-vpc-2")
    # Solo lo necesario para entrenar: PyTorch y torchvision con CUDA ya vienen instalados.
    run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "roboflow", "kagglehub", "opencv-python-headless",
                 "pycocotools"])

    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        else:
            from google.colab import userdata
            os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        print("Aviso: no se pudo leer el Secret ROBOFLOW_API_KEY:", exc)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import torch
print("Directorio:", Path.cwd(), "| Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

$ git clone -q -b medidores-energia https://github.com/martinlacheski/ceia-vpc-2.git
$ /usr/bin/python3 -m pip install -q ultralytics roboflow kagglehub opencv-python-headless pycocotools


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Directorio: /kaggle/working/ceia-vpc-2 | Kaggle: True | Colab: False
GPU: Tesla T4


In [3]:
# Descarga y preparación de los datos (solo los que piden los experimentos).
datasets = {e["dataset"] for e in EXPERIMENTS}
needs_toloka = bool(datasets & {"toloka", "combined", "combined_energia"})
needs_yfzw4 = bool(datasets & {"combined", "combined_energia", "digits", "digits_mix"})
if needs_toloka and needs_yfzw4:
    run([sys.executable, "scripts/download_data.py"])
elif needs_toloka:
    run([sys.executable, "scripts/download_data.py", "--only", "toloka"])
else:
    run([sys.executable, "scripts/download_data.py", "--only", "roboflow"])

if "digits_mix" in datasets:
    run([sys.executable, "scripts/download_data.py", "--only", "energia"])

if needs_toloka:
    run([sys.executable, "scripts/prepare_toloka.py"])
if "combined" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py"])
if "combined_energia" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py", "--extra", "data/ceml_energia",
         "--out", "data/processed/seg_toloka_yfzw4_energia"])
if "digits" in datasets:
    run([sys.executable, "scripts/prepare_digits.py"])
if "digits_mix" in datasets:
    run([sys.executable, "scripts/prepare_digits.py", "--src", "data/raw/roboflow_water_meter",
         "--src", "data/raw/energia/meter-digits-5srka", "--src", "data/raw/energia/meter-digits-frvqn",
         "--out", "data/processed/digits_mix"])

DATA = {
    "toloka": "data/processed/toloka_seg/data.yaml",
    "combined": "data/processed/seg_toloka_yfzw4/data.yaml",
    "combined_energia": "data/processed/seg_toloka_yfzw4_energia/data.yaml",
    "digits": "data/processed/digits_yfzw4/data.yaml",
    "digits_mix": "data/processed/digits_mix/data.yaml",
}

$ /usr/bin/python3 scripts/download_data.py
Toloka descargado en data/raw/toloka
loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to data/raw/roboflow_water_meter in yolov8::   0%|          | 0/17968 [00:00<?, ?it/s]

Extracting Dataset Version Zip to data/raw/roboflow_water_meter in yolov8:: 100%|██████████| 17968/17968 [00:02<00:00, 7609.49it/s] 


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
water-meter-dataset v3 descargado en data/raw/roboflow_water_meter
$ /usr/bin/python3 scripts/prepare_toloka.py
1244 imágenes -> data/processed/toloka_seg {'train': 870, 'val': 187, 'test': 187}
$ /usr/bin/python3 scripts/prepare_seg_combined.py --extra data/ceml_energia --out data/processed/seg_toloka_yfzw4_energia
ceml_energia: 587 fotos agregadas a train
yfzw4: 8217 agregadas, 755 con menos de 5 dígitos, 6 casi duplicadas de val/test de Toloka -> data/processed/seg_toloka_yfzw4_energia


In [4]:
# Entrenamiento, evaluación y empaquetado de cada experimento.
import json
import shutil

out_dir = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()


def package(name: str, weights: list[str]) -> str:
    """Arma <name>.zip con los pesos y experiments/<name>/ (en Kaggle: pestaña Output)."""
    staging = Path("runs/package") / name
    shutil.rmtree(staging, ignore_errors=True)
    (staging / "models").mkdir(parents=True)
    for path in weights:
        shutil.copy2(path, staging / "models")
    shutil.copytree(f"experiments/{name}", staging / "experiments" / name)
    return shutil.make_archive(str(out_dir / name), "zip", root_dir=staging)


for exp in EXPERIMENTS:
    print(f"\n===== {exp['name']} =====")
    trainer = exp.get("trainer", "yolo")
    if trainer == "eval_coco":
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "yolo",
             "--weights", f"models/{exp['name']}.pt", "--imgsz", str(exp["imgsz"])])
        print("Resultados:", package(exp["name"], []))
        continue
    if trainer == "maskrcnn":
        root = str(Path(DATA[exp["dataset"]]).parent)
        run([sys.executable, "scripts/train_maskrcnn.py", "--data", root, "--name", exp["name"],
             "--epochs", str(exp["epochs"]), "--batch", str(exp["batch"]), "--lr", str(exp["lr"])])
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "maskrcnn",
             "--weights", f"models/{exp['name']}.pth"])
        print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pth"]))
        continue

    run([sys.executable, "scripts/train_yolo.py", "--task", exp["task"], "--model", exp["model"],
         "--data", DATA[exp["dataset"]], "--name", exp["name"], "--imgsz", str(exp["imgsz"]),
         "--epochs", str(exp["epochs"]), "--patience", str(exp["patience"]), "--batch", str(exp["batch"]),
         "--fliplr", str(exp["fliplr"])])
    if exp["task"] == "segment":
        run([sys.executable, "scripts/evaluate_seg.py", "--model", f"models/{exp['name']}.pt",
             "--imgsz", str(exp["imgsz"])])
    else:
        # Dígitos: métricas sobre el split de prueba de yfzw4, globales y por clase.
        from ultralytics import YOLO
        res = YOLO(f"models/{exp['name']}.pt").val(data=DATA[exp["dataset"]], split="test",
                                                   imgsz=exp["imgsz"], verbose=False)
        evaluation = {
            "test": {"map50": round(float(res.box.map50), 4), "map50_95": round(float(res.box.map), 4),
                     "precision": round(float(res.box.mp), 4), "recall": round(float(res.box.mr), 4)},
            "map50_95_por_clase": {res.names[int(c)]: round(float(res.box.maps[int(c)]), 4)
                                    for c in res.box.ap_class_index},
            "imgsz": exp["imgsz"],
        }
        Path(f"experiments/{exp['name']}/eval.json").write_text(json.dumps(evaluation, indent=2))
        print(json.dumps(evaluation, indent=2))
    print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pt"]))


===== toloka_yfzw4_energia_yolov8n_seg =====
$ /usr/bin/python3 scripts/train_yolo.py --task segment --model yolov8n-seg.pt --data data/processed/seg_toloka_yfzw4_energia/data.yaml --name toloka_yfzw4_energia_yolov8n_seg --imgsz 640 --epochs 40 --patience 10 --batch 16 --fliplr 0.5
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/seg_toloka_yfzw4_energia/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fra

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/40      3.13G      1.332      1.767     0.9478      1.296          0         22        640: 100% ━━━━━━━━━━━━ 605/605 5.5it/s 1:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.2it/s 1.8s
                   all        187        187      0.961      0.927      0.967      0.562      0.907      0.866      0.879      0.524

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/40      3.13G      1.298      1.745     0.9081      1.269          0         19        640: 100% ━━━━━━━━━━━━ 605/605 5.5it/s 1:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.2it/s 1.9s
                   all        187        187   